## Likelihood analysis

In the annotation notebook, we have seen that we can add the likelihood as an annotation to each text signal. We can call the same function also to create a likelihood evaluation for each utterance in a sequence and the interations overall. This approach was proposed by:

```Mehri, S., & Eskenazi, M. (2020). USR: An unsupervised and reference free evaluation metric for dialog generation. arXiv preprint arXiv:2005.00456.```

They finetuned a RoBERTa model (USR) specifically to mimic human evaluation, as a reference-free approach. Reference-free means that is does not need a human-created response to compare the system response. Instead it determines the likelihood of the system response given the preceding conversation. The likelihood is calculated by taking the average likelihood of all tokens of the system response according to the model. The USR model was finetuned with conversations from TopicalChat and PersonaChat:

```Karthik Gopalakrishnan, Behnam Hedayatnia, Qinlang Chen, Anna Gottardi, Sanjeev Kwatra, Anu Venkatesh, Raefer Gabriel, Dilek Hakkani-Tur, and Amazon Alexa AI. 2019. Topical-chat: Towards knowledge-grounded open-domain conversations. Proc. Interspeech 2019, pages 1891–1895.```
```Saizheng Zhang, Emily Dinan, Jack Urbanek, Arthur Szlam, Douwe Kiela, and Jason Weston. 2018. Personalizing dialogue agents: I have a dog, do you have pets too? arXiv preprint arXiv:1801.07243.```

Their approach has moderate correlation with human judgments: .42 (TopicalChat) and .48 (PersonaChat). The USR model can be downloaded from the course drive: 

[usr-topicalchat-roberta_ft.zip](https://drive.google.com/file/d/1ODF-trnYeWm_hSkv9-D6-xHlnM47ToBy/view?usp=share_link)

Unpack the zip file and place it anywhere on your local machine. Adapt the path in this notebook below to your local copy of the model:

You can use any other BERT or RoBERTa model (ENCODER) from [hugggingface.co](https://huggingface.co) to score the likelihood. 


Each system token of the uterance under consideration is turned into a masked token to predict the most probable tokens according to the model (cut-off by ```len_top_tokens```). If the system token is in the list, it will receive the score from the model. If it is not in the list, the token scores ```0```.

## Prerequisites

This notebooks relies on the transformers package and the EMISSOR package for loading the EMISSOR scenarios. These packages can be installed through ```pip```:

In [1]:
# !pip install emissor
# !pip install numpy==1.26.4
# !pip install torch==2.2.2
# !pip install transformers==4.51.3

## Loading an ENCODER model for the masked-task from huggingface or disk

In [13]:
import re
from transformers import pipeline

model_name = "google-bert/bert-base-uncased"
#model_name = "FacebookAI/xlm-roberta-base"
#model_name = "FacebookAI/roberta-base"

# Path to your local copy of the USR model
#model_name ="../../models/usr-topicalchat-roberta_ft"
model = pipeline("fill-mask", model=model_name)
tokenizer = model.tokenizer

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
def mask_target_sentence(context, target):
    masked_targets = []
    target_tokens = re.split(' ', target)
    context_tokens = re.split(' ', context)
    ## We limit the length of the target as too long utterance break the token limit
    if len(context_tokens)>250:
        context_tokens = context_tokens[:250]
        print('Cut off to 250', context_tokens)
        context = ""
        for context_token in context_tokens:
            context += context_token+" "
        context = context.strip()
    if len(target_tokens)>250:
        target_tokens = target_tokens[:250]
        print('Cut off to 250', target_tokens)
    for index, token in enumerate(target_tokens):
        sequence = context + " "
        for token in target_tokens[:index]:
            sequence += token + " "
        sequence += tokenizer.mask_token
        for token in target_tokens[index + 1:]:
            sequence += " " + token
        masked_targets.append(sequence)
    return masked_targets, target_tokens

def sentence_likelihood(context, target):
    masked_targets, target_tokens = mask_target_sentence(context, target)
    expected_target = ""
    max_scores = []
    scores = []
    for masked_target, token in zip(masked_targets, target_tokens):
        results = model(masked_target)
        expected_target += results[0]['token_str'] + " "
        max_scores.append(results[0]['score'])
        match = False
        for result in results:
            if result['token_str'].lower().strip() == token.lower():
                scores.append(result['score'])
                match = True
                break
        if not match:
            scores.append(0)
    likelihood = sum(scores) / len(scores)
    max_likelihood = sum(max_scores) / len(max_scores)

    return likelihood, expected_target, max_likelihood

def score_pairs_for_likelihood(turns: []):
    for context, target in turns:
        llh, best_sentence, max_score = sentence_likelihood(context, target)
        print('Likelihood:', llh, 'Max score:', max_score, 'Best sentence:', best_sentence)

## Obtaining the conversations from EMISSOR

The next functions get the text signals from all conversations captured in the EMISSOR scenarios of a folder. Every subfolder of ```EMISSOR``` that contains a scenario JSON file is processed.

In [ ]:
import sys
from pathlib import Path

# Jupyter runs the notebook in its own folder. We add the root of the project to the path,
# so that the modules can be imported from src.cltl.emissor_evaluation
PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "cltl").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [15]:
import os
import pandas as pd
from src.cltl.emissor_evaluation.emissor import emissor_util as util

In [16]:
EMISSOR="/Users/piek/Desktop/Leolani/cltl-apps/cltl-apps-restructure_f_server/storage/emissor"

def get_scenario_ids(emissor_folder):
    scenario_ids = []
    for name in sorted(os.listdir(emissor_folder)):
        scenario_folder = os.path.join(emissor_folder, name)
        if os.path.isdir(scenario_folder) and os.path.exists(os.path.join(scenario_folder, name + ".json")):
            scenario_ids.append(name)
    return scenario_ids

def likelihood_for_scenario(emissor_folder, scenario_id):
    text_signals = util.get_text_signals_from_a_scenario(emissor_folder, scenario_id)
    print('There are:', len(text_signals), 'turns in this scenario:', scenario_id)
    llh_results = []
    context = ""
    for index, text_signal in enumerate(text_signals):
        print(f"Processing turn {index}/{len(text_signals) - 1}")
        speaker = util.get_speaker_from_text_signal(text_signal)
        text = text_signal.text
        llh, model_sentence, max_score = sentence_likelihood(context, text)
        row = {"Scenario": scenario_id, "Turn": text_signal.id, "Speaker": speaker, "Response": text, "llh": llh, "model_turn": model_sentence, "max_llh": max_score}
        llh_results.append(row)
        context = text
    return llh_results

SCENARIOS = get_scenario_ids(EMISSOR)
print('Found', len(SCENARIOS), 'scenarios in', EMISSOR)

Found 34 scenarios in /Users/piek/Desktop/Leolani/cltl-apps/cltl-apps-restructure_f_server/storage/emissor


In [17]:
all_llh_results = []
for SCENARIO in SCENARIOS:
    try:
        llh_results = likelihood_for_scenario(EMISSOR, SCENARIO)
    except Exception as e:
        print('Skipping scenario', SCENARIO, 'because of error:', e)
        continue
    if not llh_results:
        continue
    all_llh_results.extend(llh_results)

    df = pd.DataFrame(llh_results)
    evaluation_folder = os.path.join(EMISSOR, SCENARIO, 'evaluation')
    os.makedirs(evaluation_folder, exist_ok=True)
    file_name = SCENARIO+"_likelihood_evaluation.csv"
    df.to_csv(os.path.join(evaluation_folder, file_name), sep=";", index=False)

all_df = pd.DataFrame(all_llh_results)
all_df.to_csv(os.path.join(EMISSOR, "all_scenarios_likelihood_evaluation.csv"), sep=";", index=False)

There are: 37 in this scenario: 102c9186-ae30-4bcd-8c89-bd85ee899eae
Processing turn 0/36
Processing turn 1/36
Processing turn 2/36
Processing turn 3/36
Processing turn 4/36
Processing turn 5/36
Processing turn 6/36
Processing turn 7/36
Processing turn 8/36
Processing turn 9/36
Processing turn 10/36
Processing turn 11/36
Processing turn 12/36
Processing turn 13/36
Processing turn 14/36
Processing turn 15/36
Processing turn 16/36
Processing turn 17/36
Processing turn 18/36
Processing turn 19/36
Processing turn 20/36
Processing turn 21/36
Processing turn 22/36
Processing turn 23/36
Processing turn 24/36
Processing turn 25/36
Processing turn 26/36
Processing turn 27/36
Processing turn 28/36
Processing turn 29/36
Processing turn 30/36
Processing turn 31/36
Processing turn 32/36
Processing turn 33/36
Processing turn 34/36
Processing turn 35/36
Processing turn 36/36
There are: 0 in this scenario: 11315b43-fc0f-4e8f-9df2-b8382a749d47
There are: 1 in this scenario: 11807529-e689-4d73-966f-20ce

Each conversation is saved to a CSV file in the ```evaluation``` subfolder of its scenario, and all conversations together are saved to ```all_scenarios_likelihood_evaluation.csv``` in the ```EMISSOR``` folder. It has the turns on each row with the scenario identifier, the turn identifier, the speaker and the reponse.
Furthermore, it has columsn for the likelihood score (llh), the token sequence with the highest probablity according to the model (model_turn) and the maximal likelihood the model would return for the response with the highest probability tokens (max_llh), which represents the upper celing for the model:

```
Turn	Speaker	Response	llh	model_turn	max_llh
fa29f41f-7bea-42b1-bc64-17c6a1bd320e	LEOLANI	Yo Do you want to talk to me Luis?	0.661853857	 .  do  you  want  to  talk  to  me  ? 	0.786302081
4df4c74e-44cf-471a-b0cb-5233d96dbb21	SPEAKER	Yes	0	 :) 	0.118416406
d9f4848a-87ab-4599-b6ef-e5bff1149e26	LEOLANI	I have nothing more to say.	0.560176643	 i  have  much  else  to  say 	0.676033412
f37db5d2-63b5-4a49-8a5f-8825c31aafc0	SPEAKER	I live in Amstelveen.	0.455679432	 i  am  in  dc 	0.670751117
```

Since we score both the user (SPEAKER) and the agent (LEOLANI), we can obtain the likelihood of each's responses per scenario and overall, and check if the user submitted more likely responses than the agent.

In [18]:
all_df["Role"] = all_df["Speaker"].apply(lambda s: "agent" if s == "LEOLANI" else "speaker")
per_scenario = all_df.pivot_table(index="Scenario", columns="Role", values=["llh", "max_llh"], aggfunc="mean")
per_scenario.columns = [f"average_{role}_{metric}" for metric, role in per_scenario.columns]
per_scenario.to_csv(os.path.join(EMISSOR, "all_scenarios_likelihood_averages.csv"), sep=";")
per_scenario

,average_agent_llh,average_speaker_llh,average_agent_max_llh,average_speaker_max_llh
Scenario,,,,
102c9186-ae30-4bcd-8c89-bd85ee899eae,0.390840,0.030536,0.720152,0.629343
11807529-e689-4d73-966f-20cec5bdc29e,0.720191,NaN,0.937422,NaN
182f82be-2fc9-4d3b-8b45-dc2cb171e0be,0.374050,0.117832,0.750822,0.657303
219385a7-3305-4cb2-a91e-addd69ffdc92,0.372848,0.029312,0.680631,0.670828
32df20c6-3ca2-456b-9c72-d941dc454db3,0.422669,0.114019,0.735860,0.668600
5a55e02f-e2be-4dce-a9b6-bb3c6dba1480,0.424791,0.189969,0.697667,0.678750
69a394f8-e0cc-4cc5-aa33-eb143929fa44,0.730522,NaN,0.949537,NaN
79a8072c-aa20-4135-a01a-d31462b90447,0.447071,0.195079,0.754239,0.711725
91148e0c-d4b7-420e-ad2d-0dd61ffac278,0.426069,0.174099,0.722356,0.595423


In [19]:
overall = all_df.groupby("Role")[["llh", "max_llh"]].mean()
print('average_speaker_llh', overall.loc["speaker", "llh"])
print('average_speaker_max_llh', overall.loc["speaker", "max_llh"])
print('average_agent_llh', overall.loc["agent", "llh"])
print('average_agent_max_llh', overall.loc["agent", "max_llh"])

average_speaker_llh 0.13278595587832462
average_speaker_max_llh 0.6578461590783589
average_agent_llh 0.4099594058778621
average_agent_max_llh 0.7284846561785611


In this example, we see that the speaker has an average score of 0.25 and the agent a higher score of 0.28. What does that tell you?

## End of notebook